# Survival Analysis After Endodontic Treatment

A clean, refactored notebook for endodontic outcomes analysis.

The unit of analysis is an **episode** (one per `Patient_ID × Tooth_Num × Cohort`). Cohorts are *Root canal treatment*, *Root canal retreatment*, and *Apicoectomy*. The notebook builds episodes, adds coronal-restoration indicators within a 365-day post-start window, and runs descriptive, chi-square / Fisher, log-rank, Kaplan–Meier, and Cox proportional-hazards analyses, plus a PROBE / reviewer audit.

**Reusable logic lives in `analysis_final_utils.py`** (imported below as `utils`). Function names referenced in the prose match the names in the module.

> ⚠️ **Critical study parameter — `STUDY_END = 2021-01-01`.**
> The upstream SQL extract limits records to `DueDate <= '2021-01-01'`. Do **not** change this date; doing so would create artificial follow-up beyond the data lock.


## 1. Setup

Import `pandas`, `numpy`, `matplotlib`, and the reusable utilities. The utility module also re-exports `lifelines` and `scipy.stats`, so the notebook itself only needs the basics.


In [ ]:
# If needed (run once):
# !pip install lifelines openpyxl scipy


In [ ]:
import importlib
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import analysis_final_utils as utils
utils = importlib.reload(utils)


## 2. Parameters

`DATA_PATH` is the only path you should normally edit. Survival-column names, `STUDY_END`, `WINDOW_DAYS`, `COHORT_ORDER`, `RESTO_ORDER`, and the Hebrew text patterns all come from `utils` and are intentionally kept as a single source of truth.


In [ ]:
# Edit DATA_PATH to match the location of RCT_data.xlsx on your machine.
DATA_PATH = r"C:\Users\cahana_s\research\research_RCT_maria\data_old\RCT_anon.parquet"

# Re-bind utils constants for readability in the notebook.
ID_COL      = utils.ID_COL          # "Patient_ID"
TIME_COL    = utils.TIME_COL        # "duration_days"
EVENT_COL   = utils.EVENT_COL       # "event"
COHORT_COL  = utils.COHORT_COL      # "Cohort"

STUDY_END   = utils.STUDY_END       # pd.Timestamp("2021-01-01") -- DO NOT CHANGE
WINDOW_DAYS = utils.WINDOW_DAYS     # 365

COHORT_ORDER = utils.COHORT_ORDER
RESTO_ORDER  = utils.RESTO_ORDER

print("STUDY_END   :", STUDY_END.date())
print("WINDOW_DAYS :", WINDOW_DAYS, "days")


## 3. Load Data

Read the Excel file once. Patient IDs are hidden in the displayed preview.


In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "DATA_PATH not found. Please edit DATA_PATH to point to the Excel file on your machine."
    )

raw.to_parquet(PQ_PATH, index=False, engine="pyarrow")

print("Raw shape:", raw.shape)
print("Raw columns (first 30):", list(raw.columns)[:30])
display(raw.drop(columns=[c for c in ["Patient_ID"] if c in raw.columns]).head())


## 4. Clean Raw Data

`utils.clean_raw_dataframe` parses `Treatment_Date`, `Failure_Treatment_Date`, and `First_Initial_Treatment_Date` to datetimes (day-first), coerces `Patient_ID` and `Tooth_Num` to integers, derives a row-level `failure_flag_row`, and adds the row-level coronal-restoration indicators (`Has_Sealing_row`, `Has_Crown_row`) and the cohort label (`Cohort`) from the Hebrew `סיווג` column.

Hebrew-text classification rules are preserved exactly:
- `אפיס`, `אפיסקט`, or English `apic` → **Apicoectomy**
- `חידוש` → **Root canal retreatment**
- `טיפול שורש` → **Root canal treatment**
- otherwise → `None`

Coronal-restoration row-level patterns:
- sealing / build-up: `איטום|מבנה`
- crown: `הכתר|כתר`


In [ ]:
raw = utils.clean_raw_dataframe(raw)

print("Row-level failures :", int(raw["failure_flag_row"].sum()))
print("Row-level sealing prevalence :", float(raw["Has_Sealing_row"].mean()))
print("Row-level crown prevalence   :", float(raw["Has_Crown_row"].mean()))


## 5. Assign Treatment Cohorts

Cohort assignment was performed inside `utils.clean_raw_dataframe` (which calls `utils.assign_cohort` row-wise on `סיווג`). The distribution below is just a sanity check.


In [ ]:
print("Rows per cohort (raw):")
display(raw["Cohort"].value_counts(dropna=False))


## 6. Build Episodes

`utils.build_episodes` produces one episode per `Patient_ID × Tooth_Num × Cohort`. The logic is preserved exactly:

- `start_date` = earliest `Treatment_Date_dt` within the group.
- `failure_date` = earliest non-null `Failure_Treatment_Date_dt` ≥ `start_date`.
- `event` = 1 if `failure_date` is non-null, else 0.
- `stop_date` = `failure_date` for events, else `STUDY_END`.
- `duration_days` = `stop_date − start_date` in days.
- Episodes with missing or negative durations are dropped.


In [ ]:
episodes = utils.build_episodes(raw, study_end=STUDY_END)

print("Episodes shape:", episodes.shape)
display(
    episodes.groupby("Cohort")["event"]
    .agg(Episodes="count", Failures="sum", FailureRate="mean")
)


## 7. Patient Covariates and Coronal Restoration Indicators

`utils.build_patient_table` produces one row per patient (Age, AgeGroup, Male, systemic flags). `utils.add_coronal_restoration_flags` aggregates the row-level sealing / crown indicators within `[start_date, min(start_date + WINDOW_DAYS, stop_date)]` and creates the three-level `Coronal_Restoration_Group` (`Neither`, `Sealing only`, `Sealing + Crown`).


In [ ]:
patient, systemic_cols = utils.build_patient_table(raw)
print("Patient table shape:", patient.shape)
print("Systemic cols      :", systemic_cols)

episodes = utils.add_coronal_restoration_flags(episodes, raw, window_days=WINDOW_DAYS)

print("\nCoronal restoration groups (episode-level):")
display(episodes["Coronal_Restoration_Group"].value_counts(dropna=False))


## 8. Build Final Analytic Dataset

Merge patient covariates into episodes (dropping `Mac_Gender` and `Age_In_Treatment` from the patient frame first, mirroring the original notebook).


In [ ]:
episodes = utils.merge_patient_into_episodes(episodes, patient)

print("Analytic dataset shape:", episodes.shape)
display(
    episodes.drop(columns=[c for c in ["Patient_ID"] if c in episodes.columns]).head()
)


## 9. Variable Inventory

`utils.build_variable_lists` returns three lists used downstream:
- **cox_covariates** — explicit covariates (or all 0/1 dummies if none provided);
- **categorical_vars** — object/categorical/bool columns plus `Cohort`;
- **binary_vars** — all 0/1 columns (excluding survival columns).

If you want to pin a specific Cox covariate list, pass `cox_covariates=[...]` to `build_variable_lists`.


In [ ]:
COX_COVARIATES = None  # set to an explicit list if you want to pin the Cox covariates

VARLISTS = utils.build_variable_lists(
    episodes, cox_covariates=COX_COVARIATES, exclude={COHORT_COL}
)
COX_VARS = VARLISTS["cox_covariates"]
CAT_VARS = VARLISTS["categorical_vars"]
BIN_VARS = VARLISTS["binary_vars"]

print("Variable inventory:")
print(f"- Cox covariates                  : {len(COX_VARS)}")
print(f"- Categorical vars (chi2/log-rank): {len(CAT_VARS)}")
print(f"- Binary vars (Table 1 %)         : {len(BIN_VARS)}")

print("\nCohort labels:")
display(episodes[COHORT_COL].value_counts(dropna=False))


## 10. Sanity Checks and Validation

Apply the consistent base filtering used across all analyses (numeric coercion, drop missing time/event, keep duration ≥ 0). Build the two sensitivity datasets:

- **first episode per patient** — for within-patient single-observation sensitivity;
- **landmark dataset** — drops failures before `WINDOW_DAYS` and resets the time origin to mitigate the time-dependent definition of the coronal-restoration window.

Note: `episodes` is reassigned to the base-filtered version, mirroring the original notebook (this avoids drift between sections).


In [ ]:
# Strip + numeric-coerce + drop missing time/event + keep duration >= 0
episodes_base = utils.apply_base_episode_filters(episodes)

print("Analysis cohort (base filters) counts:")
display(utils.analysis_counts(episodes_base, "episodes_base"))

# Reassign to keep all downstream sections in sync
episodes = episodes_base

# Sensitivity datasets
first_episode_df = utils.first_episode_per_patient(episodes)
print("Sensitivity: first episode per patient counts:")
display(utils.analysis_counts(first_episode_df, "first_episode_df"))

LANDMARK_DAYS = WINDOW_DAYS  # default to the coronal-restoration window
landmark_df = utils.landmark_dataset(episodes, landmark_days=LANDMARK_DAYS)
print(f"\nLANDMARK_DAYS (editable): {LANDMARK_DAYS}")
print("Sensitivity: landmark_df counts:")
display(utils.analysis_counts(landmark_df, "landmark_df"))


In [ ]:
# Quick validation summaries -- shape, duplicates, survival counts
print("DataFrame shapes / duplicates:")
display(pd.concat([
    utils.summarize_dataframe(raw,         "raw"),
    utils.summarize_dataframe(patient,     "patient"),
    utils.summarize_dataframe(episodes,    "episodes (analytic)"),
], ignore_index=True))

print("\nSurvival summaries:")
display(pd.DataFrame([
    utils.summarize_survival_dataset(episodes,         "episodes (analytic)"),
    utils.summarize_survival_dataset(first_episode_df, "first_episode_df"),
    utils.summarize_survival_dataset(landmark_df,      "landmark_df"),
]))

# Episode key uniqueness check
n_episodes = len(episodes)
n_keys = episodes[["Patient_ID", "Tooth_Num", "Cohort"]].drop_duplicates().shape[0]
print(f"\nEpisode-key uniqueness (Patient_ID x Tooth_Num x Cohort):  {n_keys} / {n_episodes}")
assert n_keys == n_episodes, "Episode keys are not unique!"


## 11. Cohort Description — Primary RCT

Two compact descriptive tables for the primary nonsurgical RCT cohort, restricted to `Cohort == "Root canal treatment"` per the manuscript scope:

- **Table 1.** Baseline patient, tooth, and restoration characteristics (cohort summary, patient-level, tooth-level, restoration-related).
- **Table 2.** Per-variable unadjusted and adjusted survival outcomes — N, events, survival rate (%), log-rank *p*, per-variable Cox HR (95% CI) with cluster-robust SE on `Patient_ID`, tooth-years, and incidence per 100 TY.


In [ ]:
rct = episodes[episodes[COHORT_COL] == "Root canal treatment"].copy()

utils.display_baseline_characteristics_table(rct)
utils.display_unified_screening_table(rct)


## 12. Follow-up Time Distribution — Primary RCT

Mean ± SD, median (IQR), and number of episodes per follow-up interval (`<1y`, `1–3y`, `3–5y`, `>5y`).


In [ ]:
rct = episodes[episodes["Cohort"] == "Root canal treatment"].copy()

fu = rct["duration_days"]
print("Follow-up summary — Primary RCT cohort:")
print(f"  Mean ± SD       : {fu.mean():.1f} ± {fu.std():.1f} days")
print(f"  Median (IQR)    : {fu.median():.1f} ({fu.quantile(0.25):.1f}–{fu.quantile(0.75):.1f}) days")

bins   = [0, 365, 3 * 365, 5 * 365, float("inf")]
labels = ["<1 year", "1–3 years", "3–5 years", ">5 years"]
rct["fu_group"] = pd.cut(rct["duration_days"], bins=bins, labels=labels, right=False)

fu_dist = (
    rct.groupby("fu_group", observed=True)
       .agg(Episodes=("event", "count"), Failures=("event", "sum"))
       .reset_index()
)
fu_dist["Failure rate (%)"] = (fu_dist["Failures"] / fu_dist["Episodes"] * 100).round(2)
print("\nEpisodes by follow-up interval:")
display(fu_dist)


## 13. Methods Audit — PROBE Cohort Flow

`utils.compute_probe_tables` rebuilds the full audit (cohort flow, diagnostic exclusions, missing-data summary, restoration counts, follow-up checks, model complete-case audit, manuscript text). Expected counts (`EPISODES = 119,762; PATIENTS = 87,185; FAILURES = 3,490; CENSORED = 116,272`) are hard-coded as a manuscript reference; the audit prints WARNINGS if the rebuilt cohort differs.


In [ ]:
# Pass base_covars so the complete-case audit uses the same covariate set
# the Cox model uses below.
base_covars_for_probe = ["Cohort", "Coronal_Restoration_Group", "AgeGroup", "Male"] + systemic_cols

probe = utils.compute_probe_tables(
    raw=raw,
    episodes=episodes,
    study_end=STUDY_END,
    base_covars=base_covars_for_probe,
    excel_output_path="probe_audit_primary_rct.xlsx",
)

# --- Header and warnings (preserve original print order) ---
print("PROBE audit - primary RCT analytical cohort")
print(f"  RCT source                          : {probe['rct_source_note']}")
if probe["raw_prep_notes"]:
    print(f"  Raw audit preparation               : {'; '.join(probe['raw_prep_notes'])}")
print(f"  Primary RCT episodes                 : {probe['rct_episode_n']:,}")
if not pd.isna(probe["rct_patient_n"]):
    print(f"  Unique patients                     : {probe['rct_patient_n']:,}")
else:
    print(f"  Unique patients                     : [not available]")
if not pd.isna(probe["rct_unique_patient_tooth_n"]):
    print(f"  Unique Patient_ID + Tooth_Num pairs : {probe['rct_unique_patient_tooth_n']:,}")
else:
    print(f"  Unique Patient_ID + Tooth_Num pairs : [not available]")
print(f"  Failures / extractions (event = 1)  : {probe['rct_failure_n']:,}")
print(f"  Censored / survived (event = 0)     : {probe['rct_censored_n']:,}")

for w in probe["warnings"]:
    print(w)

# --- Tables ---
print("\nCohort-flow table:")
display(probe["cohort_flow"])

print("Diagnostic exclusions table:")
display(probe["diagnostic_exclusions"])

print("PROBE missing-data table:")
display(probe["missing_data"])

print("Restoration-group counts:")
display(probe["restoration_counts"])

print("Follow-up checks:")
display(probe["followup_checks"])

# --- STUDY_END check (preserve original message) ---
print(f"\nSTUDY_END in notebook: {pd.Timestamp(STUDY_END).date()}")
if pd.Timestamp(STUDY_END) == pd.Timestamp("2021-01-01"):
    print("WARNING: STUDY_END is 2021-01-01. Verify that manuscript study period and follow-up wording match this data cutoff.")

# --- Model audit ---
print(f"\nModel audit basis: {probe['model_basis']}")
print(f"Model variables requested: {probe['model_vars_requested']}")
print(f"Model variables available: {probe['model_vars_available']}")
print("Model complete-case audit:")
display(probe["model_complete_case_table"])

print("Model missingness by variable:")
display(probe["model_missing_by_var"])

if probe.get("excel_output_path"):
    print(f"\nExcel audit workbook written to: {os.path.abspath(probe['excel_output_path'])}")

# --- Manuscript-ready text ---
print("\nDraft manuscript text")
print(probe["methods_cohort_text"])
print()
print(probe["methods_missing_text"])
print()
print(probe["results_flow_text"])
print()
print(probe["results_missing_text"])


## 14. Pairwise Log-rank — Restoration Groups

Counts and survival (%) per restoration group, the overall multivariate log-rank, and pairwise 2-group log-rank tests with Benjamini–Hochberg-corrected *q*-values (paper Table 3).


In [ ]:
rct = episodes[episodes["Cohort"] == "Root canal treatment"].copy()

grp_summary, lr_all = utils.multi_group_logrank_summary(rct, "Coronal_Restoration_Group")
print("Log-rank test (overall, Coronal_Restoration_Group):")
print(f"  chi2 = {lr_all.test_statistic:.3f},  df = {lr_all.degrees_of_freedom},  p = {lr_all.p_value:.4e}")
display(grp_summary)

pairwise_df = utils.pairwise_logrank_with_bh(rct, "Coronal_Restoration_Group")
print("\nPairwise log-rank comparisons (with BH / FDR correction):")
display(pairwise_df)


## 15. Kaplan–Meier Curves

`utils.km_plot` converts time from days to years (÷ 365.25) before plotting (preserved exactly from the original notebook).


In [ ]:
# Overall by cohort
utils.km_plot(episodes, "Cohort",
              title="Survival by treatment type – Endodontic therapy",
              figsize=(8, 5))

# Overall by coronal restoration group
utils.km_plot(episodes, "Coronal_Restoration_Group",
              title="Survival by coronal restoration – Endodontic therapy",
              order=RESTO_ORDER, figsize=(8, 5))

# Within each cohort: coronal restoration groups
for cohort, sub in episodes.groupby("Cohort"):
    if sub["Coronal_Restoration_Group"].nunique() < 2:
        continue
    utils.km_plot(sub, "Coronal_Restoration_Group",
                  title=f"Survival by coronal restoration – {cohort}",
                  order=RESTO_ORDER, figsize=(7, 4))


## 16. Cox Proportional Hazards Models

`utils.fit_cox` calls `utils.build_cox_matrix` to create the design matrix (one-hot encoding for `Cohort`, `Coronal_Restoration_Group`, `AgeGroup`; reference categories: `Root canal treatment`, `Neither`, `40–60`).


In [ ]:
base_covars = ["Cohort", "Coronal_Restoration_Group", "AgeGroup", "Male"] + systemic_cols

print("Overall Cox model (all cohorts):")
cph_overall, hr_overall, err = utils.fit_cox(
    episodes, base_covars, penalizer=0.1, min_positive=20
)
if err:
    print(err)
else:
    display(hr_overall)

print("\nCox models within each cohort:")
per_cohort_results = {}
for cohort, sub in episodes.groupby("Cohort"):
    print("\n" + "=" * 80)
    print(f"{cohort} | Episodes={len(sub)} | Failures={int(sub['event'].sum())}")
    covars = ["Coronal_Restoration_Group", "AgeGroup", "Male"] + systemic_cols
    cph_c, hr_c, err_c = utils.fit_cox(sub, covars)
    per_cohort_results[cohort] = (cph_c, hr_c, err_c)
    if err_c:
        print(err_c)
        continue
    display(hr_c)


## 17. Forest Plot — Primary RCT

Multivariable Cox forest plot for the primary RCT cohort, using `utils.plot_cox_forest`. The default title and reference-category footer are preserved from the original notebook.


In [ ]:
rct = episodes[episodes["Cohort"] == "Root canal treatment"].copy()
covars_rct = ["Coronal_Restoration_Group", "AgeGroup", "Male"] + systemic_cols
cph_rct, hr_rct, err_rct = utils.fit_cox(rct, covars_rct, penalizer=0.1, min_positive=20)

if err_rct:
    print("Cox model error:", err_rct)
else:
    df_fp = utils.plot_cox_forest(hr_rct)
    print("Forest plot saved: forest_plot_primary_rct.png")
    print("\nHR table used for plot:")
    display(df_fp[["Variable_clean", "HR", "95% CI (lower)", "95% CI (upper)", "p_cox"]])


## 17b. Unified Results Table — Primary RCT

Publication-ready unified table for the final analytical primary nonsurgical RCT cohort. The workflow audits existing result objects in the namespace and assembles them into a single export-ready DataFrame.


In [ ]:
# Namespace audit: find existing result objects relevant to the unified primary-RCT table.
SEARCH_TERMS = [
    "summary", "table", "results", "cox", "cph", "hazard", "logrank",
    "survival", "km", "univariate", "multivariable", "regression",
    "hr", "probe", "pairwise", "forest",
],

def _iter_search_terms(terms):
    for term in terms:
        if isinstance(term, (list, tuple, set)):
            for nested in _iter_search_terms(term):
                yield str(nested).lower()
        else:
            yield str(term).lower()

def _matches_search_terms(name: str, terms) -> bool:
    lname = str(name).lower()
    return any(term in lname for term in _iter_search_terms(terms))

def _classify_object_scope(name: str) -> str:
    lname = name.lower()
    if "rct" in lname or lname in {"lr_all", "pairwise_df", "grp_summary", "fu_dist", "table1_detailed"}:
        return "Primary RCT-specific / likely primary RCT"
    if "within" in lname or "per_cohort" in lname or lname.endswith("_tbl") or lname == "sub":
        return "Per-cohort / loop-scoped"
    return "Overall / mixed scope"

candidate_rows = []
for _name in sorted(globals()):
    if _name.startswith("_") or not _matches_search_terms(_name, SEARCH_TERMS):
        continue

    _obj = globals()[_name]
    row = {
        "object_name": _name,
        "object_type": type(_obj).__name__,
        "scope_guess": _classify_object_scope(_name),
        "shape": "",
        "columns": "",
    }

    if isinstance(_obj, pd.DataFrame):
        row["shape"] = f"{_obj.shape[0]} x {_obj.shape[1]}"
        row["columns"] = ", ".join(map(str, _obj.columns.tolist()))
    elif isinstance(_obj, pd.Series):
        row["shape"] = f"{_obj.shape[0]}"
        row["columns"] = str(_obj.name)
    elif isinstance(_obj, dict):
        row["shape"] = f"len={len(_obj)}"
        row["columns"] = ", ".join(map(str, list(_obj.keys())[:15]))
    elif isinstance(_obj, (list, tuple, set)):
        row["shape"] = f"len={len(_obj)}"
    else:
        row["shape"] = getattr(_obj, "shape", "") or ""

    candidate_rows.append(row)

namespace_audit_df = pd.DataFrame(candidate_rows)

print("Candidate notebook objects for unified results table:")
display(namespace_audit_df if not namespace_audit_df.empty else pd.DataFrame(columns=["object_name", "object_type", "scope_guess", "shape", "columns"]))

base_choice_note = pd.DataFrame(
    [
        {
            "Decision": "Primary descriptive base",
            "Chosen object": "rct_final computed from final episodes",
            "Reason": "Guarantees denominator uses only the final analytical primary RCT cohort.",
        },
        {
            "Decision": "Existing descriptive helper",
            "Chosen object": "cox_style_within (if available)",
            "Reason": "Useful for cross-checking binary covariate counts, but not sufficient alone for manuscript-ready table.",
        },
        {
            "Decision": "Existing Cox source",
            "Chosen object": "hr_rct if present, else per_cohort_results['Root canal treatment'], else safe refit",
            "Reason": "Preserves current manuscript model outputs where available while remaining robust to missing/stale kernel objects.",
        },
        {
            "Decision": "Existing log-rank source",
            "Chosen object": "Existing primary-RCT objects only",
            "Reason": "Matches the requested reuse policy and avoids introducing new unrequested log-rank recomputation.",
        },
    ]
 )

print("\nBase-object selection for the unified table:")
display(base_choice_note)

In [ ]:
# Build the unified publication-ready primary-RCT results table and export it.
EXPECTED_PRIMARY_RCT = {
    "episodes": 119762,
    "patients": 87185,
    "events": 3490,
    "censored": 116272,
}

def fmt_int(value):
    if pd.isna(value):
        return "—"
    return f"{int(value):,}"

def fmt_float(value, digits=2):
    if pd.isna(value):
        return "—"
    return f"{float(value):,.{digits}f}"

def fmt_pvalue(value):
    if value in [None, "", "—"] or pd.isna(value):
        return "—"
    value = float(value)
    if value < 0.001:
        return "<0.001"
    return f"{value:.3f}"

def fmt_hr(hr, lower, upper):
    if any(pd.isna(v) for v in [hr, lower, upper]):
        return "—"
    return f"{float(hr):.2f} ({float(lower):.2f}-{float(upper):.2f})"

def resolve_existing(name, default=None):
    return globals().get(name, default)

def map_binary_level(value, yes_label="Yes", no_label="No"):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        stripped = value.strip()
        if stripped == "":
            return np.nan
        return stripped
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return str(value)
    if numeric == 1.0:
        return yes_label
    if numeric == 0.0:
        return no_label
    return np.nan

def safe_primary_rct_cox_fit(df, include_cols):
    if not include_cols:
        return None, "No eligible Cox covariates available for the primary RCT cohort."

    fit_df = df.copy()
    if ID_COL in fit_df.columns:
        fit_df[ID_COL] = fit_df[ID_COL].astype(str)
    try:
        cph_obj, hr_tbl, err_msg = utils.fit_cox(
            fit_df, include_cols, penalizer=0.1, min_positive=20
        )
        if err_msg:
            return None, err_msg
        return hr_tbl, None
    except Exception as exc:
        return None, f"Primary RCT Cox refit failed: {exc}"

rct_final = episodes.loc[episodes[COHORT_COL].astype(str).str.strip() == "Root canal treatment"].copy()
cohort_summary = {
    "episodes": int(len(rct_final)),
    "patients": int(rct_final[ID_COL].nunique()) if ID_COL in rct_final.columns else np.nan,
    "events": int(pd.to_numeric(rct_final[EVENT_COL], errors="coerce").fillna(0).sum()) if EVENT_COL in rct_final.columns else np.nan,
}
cohort_summary["censored"] = cohort_summary["episodes"] - cohort_summary["events"]

print("Primary RCT cohort lock:")
for _key in ["episodes", "patients", "events", "censored"]:
    print(f"  {_key.capitalize():<9}: {cohort_summary[_key]:,}")

for _key, _expected in EXPECTED_PRIMARY_RCT.items():
    _observed = cohort_summary.get(_key)
    if _observed != _expected:
        print(f"WARNING: primary RCT {_key} mismatch: observed {_observed:,} vs expected {_expected:,}")

# Prefer current kernel results where possible.
existing_hr_rct = resolve_existing("hr_rct")
cox_source = "hr_rct" if isinstance(existing_hr_rct, pd.DataFrame) and not existing_hr_rct.empty else None
cox_warning = None

if cox_source is None:
    per_cohort = resolve_existing("per_cohort_results", {})
    if isinstance(per_cohort, dict) and "Root canal treatment" in per_cohort:
        _tuple = per_cohort["Root canal treatment"]
        if isinstance(_tuple, tuple) and len(_tuple) >= 2 and isinstance(_tuple[1], pd.DataFrame) and not _tuple[1].empty:
            existing_hr_rct = _tuple[1].copy()
            cox_source = "per_cohort_results['Root canal treatment']"

if cox_source is None:
    model_covars = []
    if isinstance(resolve_existing("covars_rct"), list):
        model_covars = [c for c in resolve_existing("covars_rct") if c in rct_final.columns]
    else:
        candidate_covars = ["Coronal_Restoration_Group", "AgeGroup", "Male"] + [c for c in resolve_existing("systemic_cols", []) if c in rct_final.columns]
        model_covars = [c for c in candidate_covars if c in rct_final.columns]
    existing_hr_rct, cox_warning = safe_primary_rct_cox_fit(rct_final, model_covars)
    cox_source = "safe primary-RCT refit" if isinstance(existing_hr_rct, pd.DataFrame) and not existing_hr_rct.empty else "unavailable"

logrank_source = {}
if isinstance(resolve_existing("lr_all"), object) and resolve_existing("grp_summary") is not None:
    lr_obj = resolve_existing("lr_all")
    if hasattr(lr_obj, "p_value"):
        logrank_source["Coronal_Restoration_Group"] = float(lr_obj.p_value)

# Optional derivations for jaw and tooth position.
if "Jaw" not in rct_final.columns and "Tooth_Num" in rct_final.columns:
    derived = rct_final["Tooth_Num"].apply(utils.classify_tooth)
    rct_final[["Jaw", "Tooth_Position"]] = pd.DataFrame(
        [
            ("Upper jaw" if arch == "Upper" else "Lower jaw" if arch == "Lower" else np.nan,
             "Anterior" if pos == "Anterior" else "Posterior" if pos == "Posterior" else np.nan)
            for arch, pos in derived
        ],
        index=rct_final.index,
    )
elif "Tooth_Position" not in rct_final.columns and "Position" in rct_final.columns:
    rct_final["Tooth_Position"] = rct_final["Position"]

variable_specs = [
    {
        "variable_key": "AgeGroup",
        "source_col": "AgeGroup",
        "label": "Age group",
        "levels": ["<40", "40–60", "≥60"],
        "reference": "40–60",
        "transform": lambda s: s.astype(str).where(s.notna(), np.nan),
        "cox_terms": {"<40": "AgeGroup_<40", "≥60": "AgeGroup_≥60", "40–60": None},
    },
    {
        "variable_key": "Male",
        "source_col": "Male" if "Male" in rct_final.columns else ("Sex" if "Sex" in rct_final.columns else None),
        "label": "Sex",
        "levels": ["Female", "Male"],
        "reference": "Female",
        "transform": lambda s: s.map(lambda v: map_binary_level(v, yes_label="Male", no_label="Female")),
        "cox_terms": {"Male": "Male", "Female": None},
    },
    {
        "variable_key": "Smoking",
        "source_col": "Smoking" if "Smoking" in rct_final.columns else None,
        "label": "Smoking",
        "levels": ["No", "Yes"],
        "reference": "No",
        "transform": lambda s: s.map(map_binary_level),
        "cox_terms": {"Yes": "Smoking", "No": None},
    },
    {
        "variable_key": "Diabetes",
        "source_col": "Diabetes" if "Diabetes" in rct_final.columns else None,
        "label": "Diabetes mellitus",
        "levels": ["No", "Yes"],
        "reference": "No",
        "transform": lambda s: s.map(map_binary_level),
        "cox_terms": {"Yes": "Diabetes", "No": None},
    },
    {
        "variable_key": "Cancer",
        "source_col": "Cancer" if "Cancer" in rct_final.columns else None,
        "label": "Malignancy",
        "levels": ["No", "Yes"],
        "reference": "No",
        "transform": lambda s: s.map(map_binary_level),
        "cox_terms": {"Yes": "Cancer", "No": None},
    },
    {
        "variable_key": "Hypertension",
        "source_col": "Hypertension" if "Hypertension" in rct_final.columns else None,
        "label": "Hypertension",
        "levels": ["No", "Yes"],
        "reference": "No",
        "transform": lambda s: s.map(map_binary_level),
        "cox_terms": {"Yes": "Hypertension", "No": None},
    },
    {
        "variable_key": "Biphos_use",
        "source_col": "Biphos_use" if "Biphos_use" in rct_final.columns else None,
        "label": "Antiresorptive medication use",
        "levels": ["No", "Yes"],
        "reference": "No",
        "transform": lambda s: s.map(map_binary_level),
        "cox_terms": {"Yes": "Biphos_use", "No": None},
    },
    {
        "variable_key": "Coronal_Restoration_Group",
        "source_col": "Coronal_Restoration_Group" if "Coronal_Restoration_Group" in rct_final.columns else None,
        "label": "Post-endodontic coronal restoration status",
        "levels": ["Neither", "Sealing only", "Sealing + Crown"],
        "reference": "Neither",
        "transform": lambda s: s.astype(str).where(s.notna(), np.nan),
        "cox_terms": {
            "Neither": None,
            "Sealing only": "Coronal_Restoration_Group_Sealing only",
            "Sealing + Crown": "Coronal_Restoration_Group_Sealing + Crown",
        },
    },
    {
        "variable_key": "Jaw",
        "source_col": "Jaw" if "Jaw" in rct_final.columns else None,
        "label": "Jaw",
        "levels": ["Upper jaw", "Lower jaw"],
        "reference": None,
        "transform": lambda s: s.astype(str).where(s.notna(), np.nan),
        "cox_terms": {},
    },
    {
        "variable_key": "Tooth_Position",
        "source_col": "Tooth_Position" if "Tooth_Position" in rct_final.columns else None,
        "label": "Tooth position",
        "levels": ["Anterior", "Posterior"],
        "reference": None,
        "transform": lambda s: s.astype(str).where(s.notna(), np.nan),
        "cox_terms": {},
    },
]

hr_lookup = {}
if isinstance(existing_hr_rct, pd.DataFrame) and not existing_hr_rct.empty:
    hr_work = existing_hr_rct.copy()
    needed_cols = {"Variable", "HR", "95% CI (lower)", "95% CI (upper)", "p_cox"}
    if needed_cols.issubset(hr_work.columns):
        for _, _row in hr_work.iterrows():
            hr_lookup[str(_row["Variable"])] = {
                "HR": _row["HR"],
                "lower": _row["95% CI (lower)"],
                "upper": _row["95% CI (upper)"],
                "p": _row["p_cox"],
            }
    else:
        cox_warning = "Existing primary-RCT Cox table missing expected columns; Cox values set to —."
else:
    if cox_warning is None:
        cox_warning = "Primary-RCT Cox estimates are not available in the current kernel; Cox values set to —."

rows = []
checks_rows = []
warning_messages = []
if cox_warning:
    warning_messages.append(cox_warning)

for spec in variable_specs:
    source_col = spec["source_col"]
    if source_col is None or source_col not in rct_final.columns:
        warning_messages.append(f"Skipped {spec['label']}: source column not available in the final primary RCT cohort.")
        continue

    raw_series = rct_final[source_col]
    mapped = spec["transform"](raw_series)
    mapped = mapped.where(~mapped.isin(["nan", "None", "<NA>"]), np.nan)

    missing_count = int(mapped.isna().sum())
    variable_rows = []

    for level in spec["levels"]:
        subset_mask = mapped == level
        n_level = int(subset_mask.sum())
        events_level = int(pd.to_numeric(rct_final.loc[subset_mask, EVENT_COL], errors="coerce").fillna(0).sum())
        tooth_years = float(pd.to_numeric(rct_final.loc[subset_mask, TIME_COL], errors="coerce").fillna(0).sum() / 365.25)
        survival_rate = np.nan if n_level == 0 else 100.0 * (1.0 - events_level / n_level)
        incidence_rate = np.nan if tooth_years <= 0 else 100.0 * events_level / tooth_years

        term = spec.get("cox_terms", {}).get(level)
        if spec.get("reference") == level and term is None:
            hr_str = "Reference"
            p_str = "—"
        elif term in hr_lookup:
            hr_str = fmt_hr(hr_lookup[term]["HR"], hr_lookup[term]["lower"], hr_lookup[term]["upper"])
            p_str = fmt_pvalue(hr_lookup[term]["p"])
        else:
            hr_str = "—"
            p_str = "—"

        variable_rows.append({
            "Variable": spec["label"],
            "Level": level,
            "N": fmt_int(n_level),
            "Events": fmt_int(events_level),
            "Survival rate (%)": fmt_float(survival_rate, 2),
            "Log-rank p-value": "—",
            "Adjusted Cox HR (95% CI)": hr_str,
            "Cox p-value": p_str,
            "Tooth-years": fmt_float(tooth_years, 1),
            "Extraction incidence rate per 100 tooth-years": fmt_float(incidence_rate, 2),
        })

    if variable_rows:
        pval = logrank_source.get(spec["variable_key"], np.nan)
        variable_rows[0]["Log-rank p-value"] = fmt_pvalue(pval) if not pd.isna(pval) else "—"
        for _row_index in range(1, len(variable_rows)):
            variable_rows[_row_index]["Log-rank p-value"] = ""
        rows.extend(variable_rows)

    counted_n = int(mapped.notna().sum())
    counted_events = int(pd.to_numeric(rct_final.loc[mapped.notna(), EVENT_COL], errors="coerce").fillna(0).sum())
    checks_rows.append({
        "Variable": spec["label"],
        "Source column": source_col,
        "Missing or unclassified (n)": missing_count,
        "Counted N across levels": counted_n,
        "Expected total episodes": cohort_summary["episodes"],
        "N matches total unless missing": counted_n + missing_count == cohort_summary["episodes"],
        "Counted events across levels": counted_events,
        "Expected total events": cohort_summary["events"],
        "Events match total unless missing": counted_events <= cohort_summary["events"],
    })
    if counted_n + missing_count != cohort_summary["episodes"]:
        warning_messages.append(f"Level counts for {spec['label']} do not reconcile to the primary RCT cohort size.")
    if missing_count == 0 and counted_events != cohort_summary["events"]:
        warning_messages.append(f"Event counts for {spec['label']} do not reconcile to the total primary RCT events.")

unified_results_table_primary_rct = pd.DataFrame(rows)
unified_results_table_checks = pd.DataFrame(checks_rows)

print(f"\nUnified table Cox source: {cox_source}")
print("Unified table log-rank source: existing primary-RCT objects only")
if warning_messages:
    print("\nWarnings:")
    for message in dict.fromkeys(warning_messages):
        print(f"- {message}")

print("\nUnified publication-ready results table — primary RCT cohort:")
display(unified_results_table_primary_rct)

print("\nUnified table checks / missingness:")
display(unified_results_table_checks)

unified_results_table_primary_rct.to_excel("unified_results_table_primary_rct.xlsx", index=False)
unified_results_table_primary_rct.to_csv("unified_results_table_primary_rct.csv", index=False, encoding="utf-8-sig")
with pd.ExcelWriter("unified_results_table_checks.xlsx") as writer:
    unified_results_table_checks.to_excel(writer, index=False, sheet_name="checks")
    pd.DataFrame([cohort_summary]).to_excel(writer, index=False, sheet_name="cohort_summary")

print("\nExports written:")
print("- unified_results_table_primary_rct.xlsx")
print("- unified_results_table_primary_rct.csv")
print("- unified_results_table_checks.xlsx")

## 18. Final Validation Checklist

Confirms that the refactored notebook reproduces the original analysis on this data extract.


In [ ]:
checks = []

# Critical parameter
checks.append(("STUDY_END is 2021-01-01", STUDY_END == pd.Timestamp("2021-01-01")))
checks.append(("WINDOW_DAYS is 365",      WINDOW_DAYS == 365))

# Episode shape
checks.append(("episodes has > 0 rows",                                  len(episodes) > 0))
checks.append(("episode keys (Patient_ID x Tooth_Num x Cohort) unique",
               len(episodes) == episodes[["Patient_ID","Tooth_Num","Cohort"]].drop_duplicates().shape[0]))
checks.append(("duration_days >= 0 for all episodes",                    (episodes["duration_days"] >= 0).all()))
checks.append(("event in {0, 1}",                                        set(episodes["event"].dropna().unique()).issubset({0, 1})))

# Cohort labels
checks.append(("cohorts subset of expected set",
               set(episodes["Cohort"].dropna()).issubset(set(COHORT_ORDER))))

# Restoration groups
checks.append(("restoration groups subset of expected set",
               set(episodes["Coronal_Restoration_Group"].dropna()).issubset(set(RESTO_ORDER))))

# Manuscript expected counts (only meaningful on the real dataset)
rct_n        = int((episodes["Cohort"] == "Root canal treatment").sum())
rct_failures = int(episodes.loc[episodes["Cohort"] == "Root canal treatment", "event"].sum())
rct_patients = int(episodes.loc[episodes["Cohort"] == "Root canal treatment", "Patient_ID"].nunique())
rct_censored = rct_n - rct_failures
checks.append((f"RCT episodes == {utils.EXPECTED_COUNTS['episodes']:,}", rct_n == utils.EXPECTED_COUNTS["episodes"]))
checks.append((f"RCT patients == {utils.EXPECTED_COUNTS['patients']:,}", rct_patients == utils.EXPECTED_COUNTS["patients"]))
checks.append((f"RCT failures == {utils.EXPECTED_COUNTS['failures']:,}", rct_failures == utils.EXPECTED_COUNTS["failures"]))
checks.append((f"RCT censored == {utils.EXPECTED_COUNTS['censored']:,}", rct_censored == utils.EXPECTED_COUNTS["censored"]))

# Cox design matrix
X, build_err = utils.build_cox_matrix(episodes, base_covars)
checks.append(("Cox design matrix built", build_err is None and X is not None and not X.empty))
if X is not None:
    checks.append(("Cox matrix has > 0 events",
                   int(pd.to_numeric(X[EVENT_COL], errors="coerce").fillna(0).sum()) > 0))

results = pd.DataFrame(checks, columns=["Check", "Pass"])
display(results)

n_pass = int(results["Pass"].sum())
print(f"\n{n_pass} / {len(results)} checks passed.")


## 19. Review Notes

### What changed
- All inline helper functions were moved to `analysis_final_utils.py` with docstrings and (where practical) type hints. The notebook now reads as a sequence of high-level calls + displays.
- The 1278-line PROBE / reviewer audit cell was split into named helpers (`_has_cols`, `_safe_*`, `_resolve_rct`, `_rebuild_prefilter_episode_frame`, …) and a single orchestrator `compute_probe_tables` that returns a dict; the notebook now handles the prints / `display()` calls so the original output order is preserved.
- Validation helpers (`summarize_dataframe`, `summarize_survival_dataset`) and a final validation-checklist cell were added.
- The notebook now uses `import analysis_final_utils as utils` and reads constants from the module.

### What was preserved exactly
- `STUDY_END = 2021-01-01` and `WINDOW_DAYS = 365`.
- Hebrew-text classification regexes for cohort assignment and coronal restoration (`איטום|מבנה`, `הכתר|כתר`).
- Episode construction: start = min(Treatment_Date_dt), failure = min(Failure_Treatment_Date_dt ≥ start), event/censoring/duration definitions.
- Coronal-restoration window logic and the three-level `Coronal_Restoration_Group`.
- Patient covariate construction, including the inverted hypertension flag from `Hipertonia_Yes`.
- Cox model settings: `penalizer=0.1`, `min_positive=20`, cluster-robust SE on `Patient_ID`, PH-assumption check, dummy reference categories.
- Forest-plot title with `n = 119,762 Episodes`, label map, and reference-category footer.
- PROBE `EXPECTED_COUNTS` (`119,762 / 87,185 / 3,490 / 116,272`) — these are manuscript-reference values; the audit prints WARNINGS if the rebuilt cohort differs but does not silently change them.
- The Excel output filename (`probe_audit_primary_rct.xlsx`) and sheet names.

### Known things that look odd but were intentionally not changed
- **Duplicate `chi2_or_fisher_2way` definitions in the original notebook.** Cell 27 defined a version *without* an odds ratio for the chi-square branch; cell 29 redefined it *with* an OR (Haldane-Anscombe-corrected when any cell is zero). Because cell 29 ran after cell 27 and `run_chi2_batch` resolves the name at call time, the *runtime* behavior was the cell-29 version. `utils.chi2_or_fisher_2way` is the cell-29 version. The cell-27 version is gone, but its outputs are not affected because nothing actually used them.
- **`build_variable_lists` with `cox_covariates=None`** falls back to "every 0/1 column", so `COX_VARS` ends up being the same as `BIN_VARS` minus survival columns. If you want a pinned Cox covariate list, pass it via `COX_COVARIATES` in step 9. Behavior is unchanged.
- **`fit_cox` with `cluster_col`** uses `df.loc[X.index, cluster_col]`, which assumes `X.index` ⊆ `df.index`. `build_cox_matrix` only drops rows from `df`, so this holds — but it's worth noting if you ever change the design-matrix construction.
- **`episodes = episodes_base` reassignment in step 10** is intentional — it prevents drift between sections that originally used different filter states.

### TODOs / possible future improvements (NOT applied here)
- The pairwise log-rank in step 15b enumerates groups in `pd.Series.unique()` order, which is data-dependent. It would be more reproducible to sort the levels first; this would not change the q-values themselves.
- The PROBE cohort-flow table reports `n` as a mix of `int` and `np.nan`. Coercing to a nullable `Int64` would render more cleanly. No analytical impact.
- `utils.summarize_groups` uses `.groupby(...).apply(...)` which raises a pandas `FutureWarning` on recent pandas versions. Replacing with `.apply(..., include_groups=False)` would silence the warning without changing the output.
- The forest-plot title is hard-coded to `n = 119,762 Episodes`. If the dataset changes, override the `title=` argument when calling `utils.plot_cox_forest`.

### Files
- `analysis_final.ipynb` — this notebook.
- `analysis_final_utils.py` — reusable helpers.
